# v15 · The reference model: Vigenère
`PtolemyDesktop/Kryptos/Ciphers/Vigenere.py` (git 2026-08-25). One ciphertext letter as `(w | line, position, charset)`; the tabula recta as the 26 rotations of one cycle.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
import sys
sys.path.insert(0, os.path.join(PROJ, "PtolemyDesktop"))
vg = load("PtolemyDesktop/Kryptos/Ciphers/Vigenere.py", "vg")
an = load("PtolemyDesktop/Kryptos/analysis.py", "an")

## The textbook example and the `(w, i, j, k)` view

In [3]:
ct = vg.encrypt("ATTACKATDAWN", "LEMON")
print(ct); assert ct == "LXFOPVEFRNHR" and vg.decrypt(ct, "LEMON") == "ATTACKATDAWN"
for q in vg.quaternion_trace("ATTACKATDAWN", "LEMON")[:6]: print(q)

LXFOPVEFRNHR
('L' | line=0, pos=0, key='L')
('X' | line=0, pos=1, key='E')
('F' | line=0, pos=2, key='M')
('O' | line=0, pos=3, key='O')
('P' | line=0, pos=4, key='N')
('V' | line=1, pos=0, key='L')


## The tabula recta is the cyclic group Z₂₆ acting on the alphabet
Each row is a permutation (a rotation); composing two rows is the row of the sum; so the 26 rows are a group of order 26. Checked exhaustively.

In [4]:
A = vg.ALPHABET
rows = {k: tuple((i + k) % 26 for i in range(26)) for k in range(26)}
assert all(sorted(r) == list(range(26)) for r in rows.values())
compose = lambda p, q: tuple(p[q[i]] for i in range(26))
assert all(compose(rows[a], rows[b]) == rows[(a + b) % 26] for a in range(26) for b in range(26))
print("26 rows: all permutations; row(a)∘row(b) = row(a+b) for all 676 pairs")
# as complex multiplication on the 26th roots of unity
import cmath
w = [cmath.exp(2j * cmath.pi * i / 26) for i in range(26)]
k = 5
assert all(abs(w[i] * cmath.exp(2j * cmath.pi * k / 26) - w[(i + k) % 26]) < 1e-12 for i in range(26))
print("a Caesar shift by 5 = multiplication by e^(2πi·5/26) on the 26th roots of unity")

26 rows: all permutations; row(a)∘row(b) = row(a+b) for all 676 pairs
a Caesar shift by 5 = multiplication by e^(2πi·5/26) on the 26th roots of unity
